# MNN dự báo quỹ đạo bão trên dữ liệu kiểu Hurricast (`hurricast_ibtracs_1940_6h.csv`)

Notebook này đưa bộ dữ liệu `hurricast_ibtracs_1940_6h.csv` vào mô hình MNN của dự án và làm **ablation theo nhóm đặc trưng**, tất cả đều nhân quả (không dùng thông tin tương lai).

| Phần | Nội dung |
|---|---|
| 0 | Cấu hình (L2 cố định λ = 0,0005 theo kết quả quét L2) |
| 1 | Đọc dữ liệu, khôi phục vĩ độ/kinh độ gốc, xử lý giá trị thiếu, lùi các cột bị rò rỉ |
| 2 | Kiểm tra rò rỉ của từng cột theo độ trễ |
| 3 | Cắt phân đoạn, chia tập theo cột `SPLIT` (theo năm) |
| 4 | Tạo cửa sổ đầu vào theo tên đặc trưng |
| 5–7 | Mô hình MNN, huấn luyện, kiểm thử (giống `MNN_TongHop.ipynb`) |
| 8 | Ablation: Mean, **RMSE**, Median, P95, skill so với persistence, CI95 bootstrap theo cơn bão |

**Lưu ý về dữ liệu** (đã kiểm tra trên tệp):
- `STORM_SPEED`, `DIR_SIN/COS` tại t là sai phân trung tâm trên chuỗi spline 3h của IBTrACS, chứa vị trí tương lai → chỉ dùng bản **lùi 2 bước**.
- `DLAT_3H`, `DLON_3H` tại t dùng điểm spline t−3h (phụ thuộc vị trí t+6h) → chỉ dùng bản **lùi 1 bước**.
- `WIND_1MIN`, `PRES_BEST` thiếu được ghi bằng 0 (sau chuẩn hoá) → thêm cột cờ `WIND_MISS`, `PRES_MISS`.

**Cách chạy:** đặt tệp CSV cùng thư mục (hoặc thư mục cha) rồi *Run All*. Kết quả từng lần chạy lưu trong `KetQua_Hurricast/`; chạy lại sẽ bỏ qua những lần đã có.

## Thư viện

Notebook dùng `numpy`, `pandas`, `matplotlib`, `scikit-learn` và `tensorflow` (bản 2.x, gọi qua `tf.compat.v1`).
Chạy ô dưới để xem phiên bản đang dùng:

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # tránh crash kernel do trùng thư viện OpenMP trên Windows

import time
import random
import shutil

import numpy as np
import tensorflow as tf2              # import TF trước matplotlib/sklearn
import tensorflow.compat.v1 as tf
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

tf.disable_v2_behavior()
tf2.get_logger().setLevel('ERROR')

plt.rc('font', **{'size': 12})
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)

for name, mod in [('numpy', np), ('pandas', pd), ('matplotlib', matplotlib),
                  ('sklearn', sklearn), ('tensorflow', tf2)]:
    print(f'{name:11s} {mod.__version__}')

## 0. Cấu hình chung

`QUICK = True` chạy nhanh 4 epoch, 1 seed để kiểm tra đường ống; số liệu đáng tin cần `QUICK = False`.

In [ ]:
SEED = 42
CSV_PATH = 'hurricast_ibtracs_1940_6h.csv'   # tự tìm ở thư mục notebook, rồi thư mục cha
for _p in [CSV_PATH, os.path.join('..', CSV_PATH)]:
    if os.path.exists(_p):
        CSV_PATH = _p
        break
print('Dùng dữ liệu:', os.path.abspath(CSV_PATH))
OUT_DIR = 'KetQua_Hurricast'

FS_LIST = [1]              # số bước 6h cần dự báo: 1 -> +6h, 2 -> +12h, 4 -> +24h (như Hurricast)
WINDOW_SIZE = 4            # 4 bước = 24h quá khứ
FIRST_HID = [2, 25, 8]
SECOND_HID = [2, 50, 10]
TWO_STAGE = False
MAX_EPOCHS = 200
STOP_PATIENCE = 30
L2_LAMBDA = 5e-4           # chốt theo kết quả quét L2
DROP_RATE = 0.0
SEEDS = [42, 43, 44]
N_BOOT = 2000

YEAR_MIN = 1940            # đặt 1980 để bỏ giai đoạn trước vệ tinh (áp dụng cho mọi tập)
MAX_STEP_KM = 600.0        # cắt đoạn nếu nhảy > 600 km / 6h
MIN_SEG_LEN = WINDOW_SIZE + max(FS_LIST) + 1
KM_PER_DEG = 111.195
COL_LAT_ABS = 1            # trong bảng phân đoạn: cột 0 = kinh độ, cột 1 = vĩ độ (tuyệt đối)

QUICK = False
if QUICK:
    MAX_EPOCHS, SEEDS = 4, [42]

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Dữ liệu

### 1.1 Đọc tệp và khôi phục vị trí gốc

Cột `LAT`, `LON` đã bị chuẩn hoá nên vị trí gốc được lấy lại từ `LAT_SIN/LAT_COS` và `LON_SIN/LON_COS`. Kinh độ được "nối liền" trong từng cơn bão để không nhảy 360° khi vượt kinh tuyến 180°.

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=['ISO_TIME']).sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)
df['lat'] = np.degrees(np.arctan2(df['LAT_SIN'], df['LAT_COS']))
df['lon'] = np.degrees(np.arctan2(df['LON_SIN'], df['LON_COS']))
df['lon'] = df.groupby('SID')['lon'].transform(lambda s: np.degrees(np.unwrap(np.radians(s.to_numpy()))))
df['YEAR'] = df['SEASON']
print(df.shape, '| số cơn:', df['SID'].nunique(), '| mùa', df['SEASON'].min(), '-', df['SEASON'].max())
print('Bước thời gian (giờ):', (df.groupby('SID')['ISO_TIME'].diff().dt.total_seconds() / 3600).value_counts().to_dict())
df[['SID', 'ISO_TIME', 'SPLIT', 'lat', 'lon', 'WIND_1MIN', 'PRES_BEST', 'STORM_SPEED', 'TGT_DLAT_24H']].head()

### 1.2 Giá trị thiếu và cột lùi (nhân quả)

- `WIND_MISS`, `PRES_MISS` = 1 khi giá trị đúng bằng 0 (chỗ thiếu đã bị điền 0 sau chuẩn hoá).
- `SPD_L2`, `DIRS_L2`, `DIRC_L2`: `STORM_SPEED`, `DIR_SIN`, `DIR_COS` lùi 2 bước.
- `DLAT3_L1`, `DLON3_L1`: `DLAT_3H`, `DLON_3H` lùi 1 bước.

Các dòng đầu cơn chưa có giá trị lùi được điền bằng giá trị của dòng đầu tiên. Dòng đó không bao giờ là hàng cuối của một cửa sổ (cửa sổ bắt đầu từ hàng 1, điểm neo từ hàng 4), nên không đưa thông tin tương lai vào.

In [ ]:
df['WIND_MISS'] = (df['WIND_1MIN'] == 0).astype(float)
df['PRES_MISS'] = (df['PRES_BEST'] == 0).astype(float)
g = df.groupby('SID')
for src, dst, k in [('STORM_SPEED', 'SPD_L2', 2), ('DIR_SIN', 'DIRS_L2', 2), ('DIR_COS', 'DIRC_L2', 2),
                    ('DLAT_3H', 'DLAT3_L1', 1), ('DLON_3H', 'DLON3_L1', 1)]:
    df[dst] = g[src].shift(k)
    df[dst] = df[dst].fillna(g[src].transform('first'))

dec = (df['SEASON'] // 10) * 10
display(pd.DataFrame({'% WIND thiếu': 100 * df['WIND_MISS'].groupby(dec).mean(),
                      '% PRES thiếu': 100 * df['PRES_MISS'].groupby(dec).mean()}).round(1).T)

## 2. Kiểm tra rò rỉ theo độ trễ

Với mỗi cột và độ trễ k: lấy phần dư của đặc trưng sau khi trừ phần giải thích được bằng các bước dời quá khứ (hồi quy tuyến tính), rồi đo tương quan với phần dư của **bước tương lai** j → j+1. Giá trị ≈ 0 nghĩa là đặc trưng không chứa thông tin về vị trí tương lai.

In [ ]:
def step_xy(la1, lo1, la2, lo2):
    latm = np.radians((la1 + la2) / 2)
    return (lo2 - lo1) * KM_PER_DEG * np.cos(latm), (la2 - la1) * KM_PER_DEG


def resid(y, Xcols):
    X = np.c_[np.ones(len(y)), np.column_stack(Xcols)]
    return y - X @ np.linalg.lstsq(X, y, rcond=None)[0]


S = {k: step_xy(g['lat'].shift(k + 1), g['lon'].shift(k + 1), g['lat'].shift(k), g['lon'].shift(k))
     for k in range(-1, 6)}
cs = np.hypot((S[0][0] + S[-1][0]) / 2, (S[0][1] + S[-1][1]) / 2)
ok = cs.notna()
coef = np.polyfit(df.loc[ok, 'STORM_SPEED'], cs[ok], 1)          # STORM_SPEED chuẩn hoá -> km/6h
sp = np.polyval(coef, df['STORM_SPEED'])
FEATS_TEST = {'STORM_SPEED/DIR': (sp * df['DIR_SIN'], sp * df['DIR_COS']),
              'DLON_3H/DLAT_3H': (df['DLON_3H'], df['DLAT_3H'])}
rows = []
for name, F in FEATS_TEST.items():
    for lag in [0, 1, 2]:
        row = {'Đặc trưng': name, 'Độ trễ k': lag}
        for comp, cn in [(0, 'Đông'), (1, 'Bắc')]:
            f = F[comp].groupby(df['SID']).shift(lag) if lag else F[comp]
            past = [S[k][comp] for k in range(0, lag + 3)]
            m = f.notna() & S[-1][comp].notna()
            for p in past:
                m &= p.notna()
            P = [p[m].to_numpy() for p in past]
            row[f'corr với bước j→j+1 ({cn})'] = np.corrcoef(resid(f[m].to_numpy(), P),
                                                           resid(S[-1][comp][m].to_numpy(), P))[0, 1]
        rows.append(row)
display(pd.DataFrame(rows).round(3))

## 3. Cắt phân đoạn và chia tập

Mỗi cơn bão được cắt ở chỗ nhảy hơn `MAX_STEP_KM`; phân đoạn ngắn hơn `MIN_SEG_LEN` bị bỏ. Tập train/val/test lấy theo cột `SPLIT` có sẵn (chia theo năm, không cơn nào nằm ở hai tập).

Mỗi phân đoạn thành một bảng: cột 0 = kinh độ, cột 1 = vĩ độ, các cột sau là đặc trưng theo danh sách `COLS`.

In [ ]:
BASINS = ['BASIN_NA', 'BASIN_EP', 'BASIN_WP', 'BASIN_NI', 'BASIN_SI', 'BASIN_SP']      # bỏ SA (gần như rỗng)
NATURES = ['NATURE_DS', 'NATURE_TS', 'NATURE_ET', 'NATURE_SS', 'NATURE_NR', 'NATURE_MX']
COLS = (['lon', 'lat', 'LAT_SIN', 'LAT_COS', 'LON_SIN', 'LON_COS', 'DATE_SIN', 'DATE_COS'] + BASINS +
        ['WIND_1MIN', 'PRES_BEST', 'WIND_MISS', 'PRES_MISS', 'DIST2LAND'] + NATURES +
        ['SPD_L2', 'DIRS_L2', 'DIRC_L2', 'DLAT3_L1', 'DLON3_L1',
         'STORM_SPEED', 'DIR_SIN', 'DIR_COS', 'DLAT_3H', 'DLON_3H'])        # 5 cột cuối: bản rò rỉ, chỉ để đối chứng
COL = {c: i for i, c in enumerate(COLS)}


def haversine_distance(coords_true, coords_pred):
    R = 6371.0
    lon1, lat1 = np.radians(coords_true[:, 0]), np.radians(coords_true[:, 1])
    lon2, lat2 = np.radians(coords_pred[:, 0]), np.radians(coords_pred[:, 1])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


tables, meta = [], []
n_cut = 0
for sid, grp in df[df['SEASON'] >= YEAR_MIN].groupby('SID', sort=False):
    lon, lat = grp['lon'].to_numpy(), grp['lat'].to_numpy()
    km = haversine_distance(np.c_[lon[:-1], lat[:-1]], np.c_[lon[1:], lat[1:]])
    breaks = np.where(km > MAX_STEP_KM)[0] + 1
    n_cut += len(breaks)
    arr = grp[COLS].to_numpy(np.float64)
    for seg_no, idx in enumerate(np.split(np.arange(len(grp)), breaks)):
        if len(idx) < MIN_SEG_LEN:
            continue
        tables.append(arr[idx])
        meta.append({'id': sid, 'year': int(grp['SEASON'].iloc[0]), 'split': grp['SPLIT'].iloc[0], 'seg': seg_no})

split = [[i for i, m in enumerate(meta) if m['split'] == s] for s in ('train', 'val', 'test')]
tr_idx, va_idx, te_idx = split
zero_init = [(0.0, 0.0)] * len(tables)          # vị trí trong bảng đã là tuyệt đối
print(f'Cắt {n_cut} chỗ nhảy > {MAX_STEP_KM:.0f} km. {len(tables)} phân đoạn.')
display(pd.DataFrame([{'Tập': s, 'Số cơn': len({meta[i]['id'] for i in idx}), 'Số phân đoạn': len(idx),
                       'Số điểm': sum(tables[i].shape[0] for i in idx),
                       'Năm': f"{min(meta[i]['year'] for i in idx)}-{max(meta[i]['year'] for i in idx)}"}
                      for s, idx in zip(('train', 'val', 'test'), split)]))

## 4. Tạo cửa sổ đầu vào

`feature_cols` là danh sách **tên** đặc trưng. Hai tên đặc biệt `dx`, `dy` là độ dời km (Đông, Bắc) của từng bước 6h, tính từ vĩ độ/kinh độ gốc; các tên khác lấy nguyên giá trị trong bảng. Nhãn là độ dời km từ điểm neo *j* tới *j + forecast_steps*.

In [ ]:
def deg_to_km(dlon, dlat, lat_ref):
    coslat = np.cos(np.radians(lat_ref))
    return dlon * KM_PER_DEG * coslat, dlat * KM_PER_DEG


def km_to_deg(dx, dy, lat_ref):
    coslat = np.cos(np.radians(lat_ref))
    return dx / (KM_PER_DEG * coslat), dy / KM_PER_DEG


def anchor_indices(n_rows, window_size, forecast_steps=1):
    return np.arange(window_size, n_rows - forecast_steps)


def data_processing(table, window_size, forecast_steps=1, feature_cols=('dx', 'dy')):
    js = anchor_indices(table.shape[0], window_size, forecast_steps)
    nf = len(feature_cols)
    if len(js) == 0:
        return np.zeros((0, window_size, nf)), np.zeros((0, 1, 2))
    X = np.empty((len(js), window_size, nf))
    Y = np.empty((len(js), 1, 2))
    for n, j in enumerate(js):
        lat_ref = table[j, COL_LAT_ABS]
        steps = np.diff(table[j - window_size: j + 1, :2], axis=0)
        dx, dy = deg_to_km(steps[:, 0], steps[:, 1], lat_ref)
        for q, c in enumerate(feature_cols):
            X[n, :, q] = dx if c == 'dx' else dy if c == 'dy' else table[j - window_size + 1: j + 1, COL[c]]
        tgt = table[j + forecast_steps, :2] - table[j, :2]
        Y[n, 0] = deg_to_km(tgt[0], tgt[1], lat_ref)
    return X, Y


def persistence_km(table, window_size, forecast_steps=1):
    js = anchor_indices(table.shape[0], window_size, forecast_steps)
    last = table[js, :2] - table[js - 1, :2]
    dx, dy = deg_to_km(last[:, 0], last[:, 1], table[js, COL_LAT_ABS])
    return forecast_steps * np.c_[dx, dy]


X0, Y0 = data_processing(tables[0], WINDOW_SIZE, 1, ('dx', 'dy', 'WIND_1MIN', 'SPD_L2'))
print('X:', X0.shape, '| Y:', Y0.shape)
print(X0[0].round(3))

## 5. Mô hình MNN

Mỗi lớp là một lớp M-neuron song tuyến tính:

$$u_{b,j,k} = \sum_{p,q} X_{b,p,q}\, W^L_{j,k,p}\, W^R_{j,k,q} + \theta_{j,k}$$

Hai lớp ẩn có LayerNorm và hàm kích hoạt algebraic $x/\sqrt{1+x^2}$; lớp ra cho ma trận 1 × 2 (dx, dy đã chuẩn hoá).

In [ ]:
def algebraic_activation(x):
    return x / tf.sqrt(1.0 + tf.square(x))


def add_layer(inputs, in_shape, out_shape, activation_function=None, use_layernorm=False,
              layer_name='m_layer', drop_rate=None):
    n1, n2 = in_shape[1], in_shape[2]
    m1, m2 = out_shape[1], out_shape[2]
    init_std = (n1 * n2) ** -0.25          # giữ phương sai đầu ra của tích song tuyến tính ~ 1

    with tf.variable_scope(layer_name):
        W = tf.Variable(tf.random.normal([m1, m2, n1], stddev=init_std), name='W_L')
        V = tf.Variable(tf.random.normal([m1, m2, n2], stddev=init_std), name='W_R')
        biases = tf.Variable(tf.zeros([1, m1, m2]), name='theta')

        u = tf.einsum('bpq,jkp,jkq->bjk', inputs, W, V) + biases

        if use_layernorm:
            mean, variance = tf.nn.moments(u, axes=[1, 2], keepdims=True)
            gamma = tf.Variable(tf.ones([1, m1, m2]), name='gamma')
            beta = tf.Variable(tf.zeros([1, m1, m2]), name='beta')
            u = gamma * ((u - mean) / tf.sqrt(variance + 1e-6)) + beta

        outputs = u if activation_function is None else activation_function(u)
        if drop_rate is not None:
            outputs = tf.nn.dropout(outputs, rate=drop_rate)
    return W, V, outputs


def build_mnn(xs, window_size, n_feat, first_hid, second_hid, drop_rate=None):
    _, _, l1 = add_layer(xs, [None, window_size, n_feat], first_hid,
                         activation_function=algebraic_activation, use_layernorm=True,
                         layer_name='layer1', drop_rate=drop_rate)
    _, _, l2 = add_layer(l1, first_hid, second_hid,
                         activation_function=algebraic_activation, use_layernorm=True,
                         layer_name='layer2', drop_rate=drop_rate)
    _, _, prediction = add_layer(l2, second_hid, [None, 1, 2], activation_function=None,
                                 use_layernorm=False, layer_name='layer3')
    return prediction

## 6. Huấn luyện

Giống `Training.py`: Adam, lr 1e-3, batch 256, clip gradient ±1, dừng sớm khi `stop_patience` epoch liền val không giảm quá 0,2%, lưu checkpoint tại val tốt nhất.

Hai thay đổi:
- Thư mục checkpoint mặc định nằm trong thư mục làm việc (notebook không có `__file__`).
- `lr_patience` trong bản gốc khai báo nhưng không dùng (lr luôn cố định). Ở đây để mặc định `None` (giữ nguyên hành vi cũ); đặt số nguyên để bật giảm lr ×0,5 khi val chững lại.

In [ ]:
def _stack(data, window_size, forecast_steps, feature_cols):
    Xs, Ys = [], []
    for table in data:
        X, Y = data_processing(table, window_size, forecast_steps, feature_cols)
        if len(X) > 0:
            Xs.append(X)
            Ys.append(Y)
    return np.concatenate(Xs, axis=0), np.concatenate(Ys, axis=0)


def training(data_train, data_val, window_size, first_hid, second_hid, seed=None, forecast_steps=1,
             feature_cols=(0, 1, 2, 3, 4), save_dir=None, two_stage=False, batch_size=256,
             max_epochs=200, learning_rate=0.001, lr_patience=None, stop_patience=30,
             min_rel_improve=2e-3, min_lr=1e-5, verbose=True, drop_rate=0.0, l2_lambda=0.0):
    tf.reset_default_graph()
    seed = 0 if seed is None else seed
    tf.set_random_seed(seed)
    rng = np.random.RandomState(seed)

    X_tr, Y_tr = _stack(data_train, window_size, forecast_steps, feature_cols)
    X_va, Y_va = _stack(data_val, window_size, forecast_steps, feature_cols)
    n_feat = X_tr.shape[2]

    scaler_X, scaler_Y = StandardScaler(), StandardScaler()
    scaler_X.fit(X_tr.reshape(-1, n_feat))
    scaler_Y.fit(Y_tr.reshape(-1, 2))
    sx = lambda X: scaler_X.transform(X.reshape(-1, n_feat)).reshape(-1, window_size, n_feat).astype(np.float32)
    sy = lambda Y: scaler_Y.transform(Y.reshape(-1, 2)).reshape(-1, 1, 2).astype(np.float32)
    X_tr_s, Y_tr_s, X_va_s, Y_va_s = sx(X_tr), sy(Y_tr), sx(X_va), sy(Y_va)
    if verbose:
        print(f'   Mẫu train: {len(X_tr_s)} | mẫu val: {len(X_va_s)} | số đặc trưng: {n_feat}')

    xs = tf.placeholder(tf.float32, [None, window_size, n_feat])
    ys = tf.placeholder(tf.float32, [None, 1, 2])
    lr_ph = tf.placeholder(tf.float32, [])
    drop_ph = tf.placeholder_with_default(0.0, [], name='drop_rate') if drop_rate > 0 else None
    prediction = build_mnn(xs, window_size, n_feat, first_hid, second_hid, drop_rate=drop_ph)
    loss = tf.reduce_mean(tf.square(ys - prediction))
    if l2_lambda > 0:
        l2 = tf.add_n([tf.nn.l2_loss(v) for v in tf.trainable_variables() if 'W_L' in v.name or 'W_R' in v.name])
        train_loss = loss + l2_lambda * l2
    else:
        train_loss = loss

    def make_train_op(var_list):
        opt = tf.train.AdamOptimizer(learning_rate=lr_ph)
        gv = opt.compute_gradients(train_loss, var_list)
        gv = [(tf.clip_by_value(g, -1.0, 1.0), v) for g, v in gv if g is not None]
        return opt.apply_gradients(gv)

    all_vars = tf.trainable_variables()
    if two_stage:
        train_ops = [make_train_op([v for v in all_vars if 'W_R' not in v.name]),
                     make_train_op([v for v in all_vars if 'W_R' in v.name])]
    else:
        train_ops = [make_train_op(all_vars)]

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), f'saved_model_{forecast_steps}')
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, 'model')

    sess = tf.Session()
    sess.run(tf.global_variables_initializer())
    saver = tf.train.Saver(tf.trainable_variables())

    def full_loss(X, Y, chunk=8192):
        tot = 0.0
        for i in range(0, len(X), chunk):
            tot += sess.run(loss, {xs: X[i:i + chunk], ys: Y[i:i + chunk]}) * len(X[i:i + chunk])
        return tot / len(X)

    train_losses, val_losses = [], []
    best_val, best_epoch, lr = np.inf, -1, learning_rate
    ref_val, since_best, since_lr = np.inf, 0, 0
    n = len(X_tr_s)

    for epoch in range(max_epochs):
        perm = rng.permutation(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            fd = {xs: X_tr_s[b], ys: Y_tr_s[b], lr_ph: lr}
            if drop_ph is not None:
                fd[drop_ph] = drop_rate
            for op in train_ops:
                sess.run(op, fd)

        tr_l, va_l = full_loss(X_tr_s, Y_tr_s), full_loss(X_va_s, Y_va_s)
        train_losses.append(tr_l)
        val_losses.append(va_l)

        if va_l < best_val:
            best_val, best_epoch = va_l, epoch
            saver.save(sess, save_path)

        if va_l < ref_val * (1.0 - min_rel_improve):
            ref_val, since_best, since_lr = va_l, 0, 0
        else:
            since_best += 1
            since_lr += 1
            if lr_patience is not None and since_lr >= lr_patience and lr > min_lr:
                lr, since_lr = max(lr * 0.5, min_lr), 0
                if verbose:
                    print(f'   epoch {epoch}: giảm lr xuống {lr:.1e}')
            if since_best >= stop_patience:
                if verbose:
                    print(f'   Dừng sớm ở epoch {epoch}: {stop_patience} epoch liền val không giảm quá '
                          f'{100 * min_rel_improve:.1f}%')
                break

        if verbose and epoch % 10 == 0:
            print(f'   epoch {epoch:3d}  train {tr_l:.5f}  val {va_l:.5f}  lr {lr:.1e}')

    sess.close()
    if verbose:
        print(f'   Best val loss {best_val:.5f} tại epoch {best_epoch}')
    return scaler_X, scaler_Y, train_losses, val_losses, best_epoch

## 7. Kiểm thử

Khôi phục checkpoint tốt nhất, dự báo từng phân đoạn và đổi ngược về km.

In [ ]:
def testing(data_test, window_size, first_hid, second_hid, scaler_X, scaler_Y, forecast_steps=1,
            feature_cols=(0, 1, 2, 3, 4), save_dir=None):
    """Trả về (Y_true_km, Y_pred_km, valid_indices)."""
    tf.reset_default_graph()
    n_feat = scaler_X.n_features_in_
    xs = tf.placeholder(tf.float32, [None, window_size, n_feat])
    prediction = build_mnn(xs, window_size, n_feat, first_hid, second_hid)

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), f'saved_model_{forecast_steps}')
    sess = tf.Session()
    tf.train.Saver(tf.trainable_variables()).restore(sess, os.path.join(save_dir, 'model'))

    Y_true, Y_pred, valid = [], [], []
    for idx, storm in enumerate(data_test):
        X, Y = data_processing(storm, window_size, forecast_steps, feature_cols)
        if len(X) == 0:
            continue
        valid.append(idx)
        X_s = scaler_X.transform(X.reshape(-1, n_feat)).reshape(-1, window_size, n_feat)
        p = sess.run(prediction, feed_dict={xs: X_s})
        Y_true.append(Y.reshape(-1, 2))
        Y_pred.append(scaler_Y.inverse_transform(p.reshape(-1, 2)))
    sess.close()
    return Y_true, Y_pred, valid


def to_abs(table, initial, disp_km, forecast_steps):
    """Độ dời km từ điểm neo -> toạ độ tuyệt đối (lon, lat)."""
    js = anchor_indices(table.shape[0], WINDOW_SIZE, forecast_steps)
    anchor = table[js, :2] + np.array(initial)
    dlon, dlat = km_to_deg(disp_km[:, 0], disp_km[:, 1], table[js, COL_LAT_ABS])
    return anchor + np.c_[dlon, dlat]


def hav_errors(tables, initials, valid, true_km, pred_km, fs):
    """Sai số Haversine (km) của mọi điểm dự báo, nối theo thứ tự phân đoạn."""
    return np.concatenate([haversine_distance(to_abs(tables[i], initials[i], t, fs),
                                              to_abs(tables[i], initials[i], p, fs))
                           for i, t, p in zip(valid, true_km, pred_km)])

## 8. Ablation theo nhóm đặc trưng

Các nhóm được thêm dần, mỗi nhóm là một loại **thông tin mới** so với dx, dy:

| Cấu hình | Thêm | Nhân quả |
|---|---|---|
| A | dx, dy | có |
| G | + vị trí tuyệt đối (sin/cos vĩ độ, kinh độ), ngày trong năm, lưu vực | có |
| GI | + gió, áp suất (kèm cờ thiếu), khoảng cách tới đất liền | có |
| GIN | + loại bão | có |
| GINS | + speed/dir lùi 2 bước, độ dời 3h lùi 1 bước | có |
| LEAK | GIN + speed/dir và độ dời 3h **tại t** (bản gốc của tệp) | **không**, chỉ để đối chiếu |

Chọn cấu hình theo **val**; tập test chỉ để báo cáo. L2 cố định λ = `L2_LAMBDA`.

In [ ]:
GEO = ['LAT_SIN', 'LAT_COS', 'LON_SIN', 'LON_COS', 'DATE_SIN', 'DATE_COS'] + BASINS
INT = ['WIND_1MIN', 'PRES_BEST', 'WIND_MISS', 'PRES_MISS', 'DIST2LAND']
SPD_CAUSAL = ['SPD_L2', 'DIRS_L2', 'DIRC_L2', 'DLAT3_L1', 'DLON3_L1']
SPD_LEAK = ['STORM_SPEED', 'DIR_SIN', 'DIR_COS', 'DLAT_3H', 'DLON_3H']

CFGS = {
    'A':    (['dx', 'dy'],                                   True,  'A: dx, dy'),
    'G':    (['dx', 'dy'] + GEO,                             True,  'G: A + vị trí tuyệt đối, ngày, lưu vực'),
    'GI':   (['dx', 'dy'] + GEO + INT,                       True,  'GI: G + gió, áp suất, khoảng cách đất liền'),
    'GIN':  (['dx', 'dy'] + GEO + INT + NATURES,             True,  'GIN: GI + loại bão'),
    'GINS': (['dx', 'dy'] + GEO + INT + NATURES + SPD_CAUSAL, True, 'GINS: GIN + speed/dir lùi 2, độ dời 3h lùi 1'),
    'LEAK': (['dx', 'dy'] + GEO + INT + NATURES + SPD_LEAK,  False, 'LEAK: GIN + speed/dir, độ dời 3h tại t (rò rỉ)'),
}
RUN_CFGS = ['A', 'G', 'GI', 'GIN', 'GINS']        # thêm 'LEAK' nếu muốn có dòng đối chiếu
COMPARISONS = [('G', 'A'), ('GI', 'G'), ('GIN', 'GI'), ('GINS', 'GIN'), ('GINS', 'A'), ('LEAK', 'GIN')]

for k in RUN_CFGS:
    print(f'{k:5s} {len(CFGS[k][0]):2d} đặc trưng: {CFGS[k][0]}')
print(f'\n{len(RUN_CFGS)} cấu hình × {len(SEEDS)} seed × {len(FS_LIST)} hạn = {len(RUN_CFGS) * len(SEEDS) * len(FS_LIST)} lần chạy, '
      f'tối đa {MAX_EPOCHS} epoch mỗi lần')

### 8.1 Huấn luyện

In [ ]:
def run_file(cfg, fs, seed):
    return os.path.join(OUT_DIR, f'{cfg}_fs{fs}_y{YEAR_MIN}_l2{L2_LAMBDA:g}_ep{MAX_EPOCHS}_s{seed}.npz')


def run_one(cfg, fs, seed):
    path = run_file(cfg, fs, seed)
    if os.path.exists(path):
        return False
    cols = CFGS[cfg][0]
    tr, va, te = ([tables[i] for i in idx] for idx in split)
    sd = os.path.join(OUT_DIR, 'ckpt', f'{cfg}_{fs}_{seed}')
    t0 = time.time()
    sX, sY, trl, vll, best = training(tr, va, WINDOW_SIZE, FIRST_HID, SECOND_HID, seed=seed, forecast_steps=fs,
                                      feature_cols=cols, save_dir=sd, two_stage=TWO_STAGE, max_epochs=MAX_EPOCHS,
                                      stop_patience=STOP_PATIENCE, drop_rate=DROP_RATE, l2_lambda=L2_LAMBDA,
                                      verbose=False)
    out = {'best_epoch': best, 'train_losses': np.array(trl), 'val_losses': np.array(vll)}
    for name, tabs in (('val', va), ('test', te)):
        yt, yp, valid = testing(tabs, WINDOW_SIZE, FIRST_HID, SECOND_HID, sX, sY, fs, cols, save_dir=sd)
        out[f'err_{name}'] = hav_errors(tabs, zero_init, valid, yt, yp, fs)
    np.savez(path, **out)
    shutil.rmtree(sd, ignore_errors=True)
    ev, et = out['err_val'], out['err_test']
    print(f'val Mean {ev.mean():6.2f} RMSE {np.sqrt(np.mean(ev ** 2)):6.2f} | '
          f'test Mean {et.mean():6.2f} RMSE {np.sqrt(np.mean(et ** 2)):6.2f} km | best_epoch {best} | {time.time() - t0:.0f}s')
    return True


jobs = [(c, fs, s) for fs in FS_LIST for s in SEEDS for c in RUN_CFGS]
for n, (c, fs, s) in enumerate(jobs, 1):
    print(f'[{n}/{len(jobs)}] {c} +{6 * fs}h seed={s}: ', end='')
    if not run_one(c, fs, s):
        print('đã có, bỏ qua')

### 8.2 Bảng kết quả

- Bảng 1: val (để chọn cấu hình, ★) và test: Mean, RMSE (kèm độ lệch chuẩn theo seed), Median, P95, skill so với persistence.
- Bảng 2: Δ = trái − phải trên test, CI95 bootstrap theo cơn bão. Δ < 0 nghĩa là cấu hình bên trái tốt hơn.

In [ ]:
def rmse(e):
    return float(np.sqrt(np.mean(e ** 2)))


def errors_persistence(idx, fs):
    tabs = [tables[i] for i in idx]
    true_km = [data_processing(t, WINDOW_SIZE, fs, ('dx',))[1].reshape(-1, 2) for t in tabs]
    per_km = [persistence_km(t, WINDOW_SIZE, fs) for t in tabs]
    return hav_errors(tabs, zero_init, range(len(tabs)), true_km, per_km, fs)


def test_groups(fs):
    ids = np.concatenate([[meta[i]['id']] * len(anchor_indices(tables[i].shape[0], WINDOW_SIZE, fs)) for i in te_idx])
    return pd.factorize(ids)[0]


def boot_delta(Ex, Ey, groups, n_boot=N_BOOT, seed=SEED):
    G = groups.max() + 1
    cnt = np.bincount(groups, minlength=G).astype(float)
    S1 = lambda E: np.stack([np.bincount(groups, e, G) for e in E])
    S2 = lambda E: np.stack([np.bincount(groups, e ** 2, G) for e in E])
    s1x, s2x, s1y, s2y = S1(Ex), S2(Ex), S1(Ey), S2(Ey)
    w = np.random.default_rng(seed).multinomial(G, np.full(G, 1.0 / G), size=n_boot).astype(float)
    nb = w @ cnt
    d_mean = ((w @ s1x.T) - (w @ s1y.T)).mean(axis=1) / nb
    d_rmse = (np.sqrt((w @ s2x.T) / nb[:, None]) - np.sqrt((w @ s2y.T) / nb[:, None])).mean(axis=1)
    pm = np.mean(Ex.mean(axis=1) - Ey.mean(axis=1))
    pr = np.mean(np.sqrt((Ex ** 2).mean(axis=1)) - np.sqrt((Ey ** 2).mean(axis=1)))
    return pm, np.percentile(d_mean, [2.5, 97.5]), pr, np.percentile(d_rmse, [2.5, 97.5])


results = {}
for fs in FS_LIST:
    groups = test_groups(fs)
    e_per_te, e_per_va = errors_persistence(te_idx, fs), errors_persistence(va_idx, fs)
    E, rows = {}, []
    for key, (cols, causal, label) in CFGS.items():
        files = [run_file(key, fs, s) for s in SEEDS if os.path.exists(run_file(key, fs, s))]
        if not files:
            continue
        runs = [np.load(f) for f in files]
        et, ev = np.stack([r['err_test'] for r in runs]), np.stack([r['err_val'] for r in runs])
        assert et.shape[1] == len(e_per_te)
        E[key] = et
        sd = (lambda v: v.std(ddof=1)) if len(runs) > 1 else (lambda v: np.nan)
        tm, tr_ = et.mean(axis=1), np.sqrt((et ** 2).mean(axis=1))
        rows.append({'Cấu hình': label, 'Nhân quả': causal, 'n_feat': len(cols), 'n_seed': len(runs),
                     'Val Mean (km)': ev.mean(axis=1).mean(), 'Val RMSE (km)': np.sqrt((ev ** 2).mean(axis=1)).mean(),
                     'Test Mean (km)': tm.mean(), 'Std Test Mean': sd(tm),
                     'Test RMSE (km)': tr_.mean(), 'Std Test RMSE': sd(tr_),
                     'Test Median (km)': np.median(et, axis=1).mean(), 'Test P95 (km)': np.percentile(et, 95, axis=1).mean(),
                     'best_epoch': np.mean([int(r['best_epoch']) for r in runs])})
    rows.append({'Cấu hình': 'Persistence', 'Nhân quả': True,
                 'Val Mean (km)': e_per_va.mean(), 'Val RMSE (km)': rmse(e_per_va),
                 'Test Mean (km)': e_per_te.mean(), 'Test RMSE (km)': rmse(e_per_te),
                 'Test Median (km)': np.median(e_per_te), 'Test P95 (km)': np.percentile(e_per_te, 95)})
    tab = pd.DataFrame(rows)
    tab['Skill Mean vs persistence (%)'] = 100 * (1 - tab['Test Mean (km)'] / e_per_te.mean())
    tab['Skill RMSE vs persistence (%)'] = 100 * (1 - tab['Test RMSE (km)'] / rmse(e_per_te))
    causal_rows = tab[(tab['Nhân quả']) & (tab['Cấu hình'] != 'Persistence')]
    tab['Chọn theo val'] = ''
    if len(causal_rows):
        tab.loc[causal_rows['Val Mean (km)'].idxmin(), 'Chọn theo val'] += '★ Mean '
        tab.loc[causal_rows['Val RMSE (km)'].idxmin(), 'Chọn theo val'] += '★ RMSE'

    comp = []
    for x, y in COMPARISONS:
        if x in E and y in E:
            dm, ci_m, dr, ci_r = boot_delta(E[x], E[y], groups)
            comp.append({'So sánh': f'{x} so với {y}', 'Δ Mean (km)': dm, 'CI95 Mean': f'[{ci_m[0]:.2f}, {ci_m[1]:.2f}]',
                         'Mean: CI loại trừ 0': bool(ci_m[0] > 0 or ci_m[1] < 0),
                         'Δ RMSE (km)': dr, 'CI95 RMSE': f'[{ci_r[0]:.2f}, {ci_r[1]:.2f}]',
                         'RMSE: CI loại trừ 0': bool(ci_r[0] > 0 or ci_r[1] < 0)})
    comp = pd.DataFrame(comp)
    tab.round(4).to_csv(os.path.join(OUT_DIR, f'Bang_cau_hinh_{6 * fs}h.csv'), index=False, encoding='utf-8-sig')
    comp.round(4).to_csv(os.path.join(OUT_DIR, f'So_sanh_cap_{6 * fs}h.csv'), index=False, encoding='utf-8-sig')
    results[fs] = (tab, comp)

    print(f'\n========== +{6 * fs}h | {len(e_per_te)} điểm test, {groups.max() + 1} cơn bão | L2 = {L2_LAMBDA:g} ==========')
    display(tab.set_index('Cấu hình').round(3))
    if len(comp):
        display(comp.set_index('So sánh').round(3))
print(f'Đã lưu CSV vào {OUT_DIR}/')

### 8.3 Biểu đồ Mean và RMSE theo cấu hình

Thanh sọc là cấu hình không nhân quả; vạch đứt là persistence; thanh lỗi là độ lệch chuẩn theo seed.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

for fs, (tab, _) in results.items():
    models = tab[tab['Cấu hình'] != 'Persistence'].reset_index(drop=True)
    pers = tab[tab['Cấu hình'] == 'Persistence'].iloc[0]
    labels = [s.split(':')[0] for s in models['Cấu hình']]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, metric in zip(axes, ['Mean', 'RMSE']):
        vals = models[f'Test {metric} (km)']
        bars = ax.bar(labels, vals, yerr=models[f'Std Test {metric}'].fillna(0), width=0.6,
                      color='#4C78A8', edgecolor='white', capsize=4)
        for b, causal in zip(bars, models['Nhân quả']):
            if not causal:
                b.set_hatch('//')
                b.set_alpha(0.55)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v / 2, f'{v:.1f}', ha='center', va='center',
                    color='white', fontsize=10, fontweight='bold')
        ax.axhline(pers[f'Test {metric} (km)'], color='#555555', ls='--', lw=1.5)
        ax.set_title(f'{metric} sai số Haversine, test +{6 * fs}h')
        ax.set_ylabel('km')
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)
    handles = [Patch(facecolor='#4C78A8', label='Nhân quả'),
               Patch(facecolor='#4C78A8', alpha=0.55, hatch='//', edgecolor='white', label='Không nhân quả (rò rỉ)'),
               Line2D([], [], color='#555555', ls='--', lw=1.5,
                      label=f'Persistence (Mean {pers["Test Mean (km)"]:.1f} | RMSE {pers["Test RMSE (km)"]:.1f} km)')]
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()

### 8.4 Cách đọc

- So sánh dọc theo chuỗi A → G → GI → GIN → GINS cho biết **mỗi nhóm thông tin mới** đóng góp bao nhiêu; chỉ coi là đóng góp thật khi CI95 của Δ loại trừ 0.
- Dòng LEAK (nếu chạy) cho thấy bản gốc của `STORM_SPEED`/`DLAT_3H` trong tệp làm sai số giảm "giả" đến mức nào; không dùng để báo cáo kết quả.
- Muốn so với Hurricast (dự báo 24h), đặt `FS_LIST = [4]`. Lưu ý Hurricast báo cáo trên Bắc Đại Tây Dương và Đông Thái Bình Dương, còn tập test ở đây gồm mọi lưu vực.